# Titanic — survival prediction

Predicting who survived the Titanic, for the
[Kaggle competition](https://www.kaggle.com/c/titanic).
Best leaderboard score: **0.80861**.

The notebook runs in order: look at the data → build the features →
measure what each feature is worth → train → predict on `test.csv`.

## 1. The data

891 training rows, 418 test rows. First: what is missing.


In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# --- load the data (paths from .env, falling back to ../data) -----------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
load_dotenv(ROOT / ".env")
DATA_DIR = Path(os.getenv("DATA_DIR", ROOT / "data"))

train = pd.read_csv(os.getenv("TRAIN_CSV", DATA_DIR / "train.csv"))
test  = pd.read_csv(os.getenv("TEST_CSV",  DATA_DIR / "test.csv"))

# --- missing values, train vs test -------------------------------------------
missing = pd.DataFrame({
    "dtype":         train.dtypes.astype(str),
    "train_missing": train.isna().sum().astype("Int64"),
    "train_pct":     (100 * train.isna().mean()).round(2),
    "test_missing":  test.isna().sum().astype("Int64"),   # NaN for Survived: not in test
    "test_pct":      (100 * test.isna().mean()).round(2),
})
missing["total_missing"] = missing[["train_missing", "test_missing"]].sum(axis=1).astype("Int64")
missing = missing.sort_values("total_missing", ascending=False)

print(f"train: {train.shape[0]} rows x {train.shape[1]} cols     "
      f"test: {test.shape[0]} rows x {test.shape[1]} cols\n")
# print with "-" wherever the column simply does not exist in that file
print(missing.astype(object).where(missing.notna(), "-").to_string())
print(f"\ncolumns with missing values : {(missing['total_missing'] > 0).sum()} of {len(missing)}"
      f"  ->  {missing.index[missing['total_missing'] > 0].tolist()}")
print(f"rows fully complete         : train {train.dropna().shape[0]}/{len(train)}, "
      f"test {test.dropna().shape[0]}/{len(test)}")
print("(Survived is '-' in test because the column does not exist there — it is the target.)")



train: 891 rows x 12 cols     test: 418 rows x 11 cols

               dtype train_missing train_pct test_missing test_pct total_missing
Cabin            str           687      77.1          327    78.23          1014
Age          float64           177     19.87           86    20.57           263
Embarked         str             2      0.22            0      0.0             2
Fare         float64             0       0.0            1     0.24             1
Name             str             0       0.0            0      0.0             0
Parch          int64             0       0.0            0      0.0             0
PassengerId    int64             0       0.0            0      0.0             0
Pclass         int64             0       0.0            0      0.0             0
Sex              str             0       0.0            0      0.0             0
SibSp          int64             0       0.0            0      0.0             0
Survived       int64             0       0.0         

`Cabin` is missing in ~77% of rows, so we drop it. `Age` is missing in ~20%
and `Embarked` in 2 rows — both get filled with a constant (see section 3).

## 2. Outliers

Are the extreme `Fare` and `Age` values errors we should remove, or real
records we should keep?


In [ ]:
NUM_COLS = ["Age", "Fare", "SibSp", "Parch"]

rows = []
for name, df in [("train", train), ("test", test)]:
    for c in NUM_COLS:
        s = df[c].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        z = (s - s.mean()) / s.std()
        n_iqr = int(((s < lo) | (s > hi)).sum())
        rows.append({
            "file": name, "col": c,
            "min": round(s.min(), 2), "max": round(s.max(), 2),
            "Q1": round(q1, 2), "Q3": round(q3, 2), "IQR": round(iqr, 2),
            "low_fence": round(lo, 2), "high_fence": round(hi, 2),
            "n_IQR_out": n_iqr, "pct_IQR": round(100 * n_iqr / len(s), 1),
            "n_z>3": int((z.abs() > 3).sum()),
            "skew": round(s.skew(), 2),
        })

outliers = pd.DataFrame(rows).set_index(["file", "col"])
print("IQR and z-score outlier counts\n")
print(outliers.to_string())

print("\n" + "-" * 70)
print("the actual extremes (train + test combined)\n")
full = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)
for c in NUM_COLS:
    top = full.nlargest(3, c)[["PassengerId", "split", "Pclass", "Name", c]]
    print(f"  highest {c}:")
    for _, r in top.iterrows():
        print(f"     {r[c]:>8}  id={r['PassengerId']:<5} {r['split']:<5} class {r['Pclass']}  {r['Name'][:45]}")
low_age = [round(float(a), 2) for a in sorted(full["Age"].dropna().unique())[:5]]
print(f"\n  lowest Age: {low_age}  <- infants stored as fractions of a year")

print("\n" + "-" * 70)
print("plausibility checks\n")
print(f"  negative values           : {int((full[NUM_COLS] < 0).sum().sum())}")
print(f"  Age > 100                 : {int((full['Age'] > 100).sum())}")
print(f"  Fare == 0                 : {int((full['Fare'] == 0).sum())}  <- free passage, real records")
print(f"  Fare == 512.33 (the max)  : {int((full['Fare'] == full['Fare'].max()).sum())} rows, "
      f"all on ticket {full.loc[full['Fare'].idxmax(), 'Ticket']} -> one group booking, not a typo")

print("\n  zero-inflation warning (the IQR rule misbehaves on count columns):")
for c in ["SibSp", "Parch"]:
    s_c = full[c]
    pct0 = 100 * (s_c == 0).mean()
    q1, q3 = s_c.quantile([0.25, 0.75])
    iqr = q3 - q1
    hi = q3 + 1.5 * iqr
    if iqr == 0:
        why = (f"Q1=Q3={q1:.0f} so IQR=0 and the fence collapses onto {hi:.0f} -> it flags "
               f"EVERY non-zero value. Verdict meaningless.")
    else:
        why = (f"Q1={q1:.0f}, Q3={q3:.0f}, so the fence is {hi:.1f} -> it flags anyone with "
               f">{hi:.1f} relatives aboard, which is a large family, not an error.")
    print(f"     {c}: {pct0:.1f}% of rows are 0. {why}")

print("\n  Fare judged within its own Pclass instead of against everyone:")
for p in sorted(full["Pclass"].unique()):
    s = full.loc[full["Pclass"] == p, "Fare"].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    hi = q3 + 1.5 * iqr
    print(f"     class {p}: median {s.median():6.2f}  fence {hi:7.2f}  "
          f"outliers {int((s > hi).sum()):3d}")

IQR and z-score outlier counts

              min     max     Q1    Q3    IQR  low_fence  high_fence  n_IQR_out  pct_IQR  n_z>3  skew
file  col                                                                                            
train Age    0.42   80.00  20.12  38.0  17.88      -6.69       64.81         11      1.5      2  0.39
      Fare   0.00  512.33   7.91  31.0  23.09     -26.72       65.63        116     13.0     20  4.79
      SibSp  0.00    8.00   0.00   1.0   1.00      -1.50        2.50         46      5.2     30  3.70
      Parch  0.00    6.00   0.00   0.0   0.00       0.00        0.00        213     23.9     15  2.75
test  Age    0.17   76.00  21.00  39.0  18.00      -6.00       66.00          2      0.6      1  0.46
      Fare   0.00  512.33   7.90  31.5  23.60     -27.51       66.91         55     13.2     18  3.69
      SibSp  0.00    8.00   0.00   1.0   1.00      -1.50        2.50         11      2.6      7  4.17
      Parch  0.00    9.00   0.00   0.0   0.00     

**Nothing is removed.** No negative values, no impossible ages. The max fare
of 512.33 is shared by several passengers on one ticket — a group booking, not
a typo — and the tiny ages are infants stored as fractions of a year. The IQR
fence is also meaningless on `SibSp`/`Parch`, where most rows are 0, so it
flags every large family as an "outlier".

That one observation — *a ticket price is charged per group, not per person* —
is what leads to the `FarePerPax` feature below.

## 3. Feature engineering

Cleaning and feature building live in `scripts/` so the notebook and the
pipeline use exactly the same code.

The 4 raw columns we keep are `Sex`, `Pclass`, `Age`, `Fare`. On top of them we
build 5 features:

| feature | how | the idea behind it |
|---|---|---|
| `Title` | parsed from `"Last, Title. First"` → Mr / Miss / Mrs / Master / Rare | "Master" marks a **boy**, which `Age` alone misses when Age is filled in |
| `FamilySize` | `SibSp + Parch + 1` | families moved together, so they lived or died together |
| `IsAlone` | `FamilySize == 1` | a solo traveller had nobody slowing them down |
| `FarePerPax` | `Fare / FamilySize` | the fare is the **whole ticket**, repeated on every member's row — dividing it gives real wealth |
| `IsChild` | `Age < 12` | "women and children first" |

Two rules we stuck to:

- **Fills are constants, not `df.median()`.** `Age` → 28.0 and `Fare` → 14.4542,
  both the *train* medians. Computing the median per file would fill the 86 test
  rows missing Age with 27.0 — a value the model never trained on.
- **Every feature is computed from a single row.** Nothing uses a count or a
  median over the whole file, so train and test are always built identically.


In [3]:
import sys
sys.path.append(str(ROOT / "scripts"))

from cleaning import clean
from feature_engineering import build_features

train_clean = clean(train.copy())          # drop Cabin, fill Age and Embarked
X, y = build_features(train_clean)         # 4 raw + 5 engineered columns

print(f"{X.shape[1]} features for {X.shape[0]} passengers:  {list(X.columns)}\n")
print(X.head().to_string())

9 features for 891 passengers:  ['Sex', 'Pclass', 'Age', 'Fare', 'Title', 'FamilySize', 'IsAlone', 'FarePerPax', 'IsChild']

   Sex  Pclass   Age     Fare  Title  FamilySize  IsAlone  FarePerPax  IsChild
0    0       3  22.0   7.2500      0           2        0     3.62500        0
1    1       1  38.0  71.2833      2           2        0    35.64165        0
2    1       3  26.0   7.9250      1           1        1     7.92500        0
3    1       1  35.0  53.1000      2           2        0    26.55000        0
4    0       3  35.0   8.0500      0           1        1     8.05000        0


### What each feature is actually worth

Adding features one group at a time and cross-validating each time
(5 folds x 3 repeats — a single split on 891 rows swings ~4 points, which is
wider than most of these differences).


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
def forest():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42)

steps = {
    "Sex alone":                    ["Sex"],
    "+ Pclass, Age, Fare":          ["Sex", "Pclass", "Age", "Fare"],
    "+ Title":                      ["Sex", "Pclass", "Age", "Fare", "Title"],
    "+ FamilySize, IsAlone":        ["Sex", "Pclass", "Age", "Fare", "Title", "FamilySize", "IsAlone"],
    "+ FarePerPax, IsChild (all)":  list(X.columns),
}

previous = None
for name, cols in steps.items():
    score = cross_val_score(forest(), X[cols], y, cv=cv, n_jobs=-1).mean()
    delta = "" if previous is None else f"   {score - previous:+.4f}"
    print(f"{name:32s} {score:.4f}{delta}")
    previous = score

Sex alone                        0.7867


+ Pclass, Age, Fare              0.8301   +0.0434


+ Title                          0.8414   +0.0112


+ FamilySize, IsAlone            0.8316   -0.0097


+ FarePerPax, IsChild (all)      0.8395   +0.0078


**Sex alone already gets 0.7867.** That is the bar every extra feature has to
beat, and the numbers say plainly where the gains are:

- `Pclass, Age, Fare` are the big jump (**+0.0434**) — your class and your fare.
- `Title` adds **+0.0112**, mostly by separating `Master` (a boy) from `Mr`.
- `FamilySize, IsAlone` **cost 0.0097**. Handed to the trees as plain columns,
  they are just two more numbers to memorise.
- `FarePerPax, IsChild` bring back **+0.0078**, ending at **0.8395** for all 9.

So the family idea is worth little *inside* the model — yet it is the strongest
signal available. Section 5 shows how to use it properly: applied directly,
not fed to the forest.

### What did *not* help

Measured and dropped, so they are not in the code:

- `Sex * Age` — 0.87 correlated with `Sex`; pure memorisation surface.
- `Embarked`, raw `SibSp`/`Parch`, `NameLength`, `log(Fare)`, ticket prefixes — neutral or negative.
- Binning `Fare` or `Age` — reduced overfitting but cost accuracy.
- Filling `Age` per title instead of one median — worth +0.03 points.

## 4. Training

A random forest with `min_samples_leaf=3`. That one setting is the whole
anti-overfitting knob: at the default of 1 the trees reach ~0.98 on data they
have already seen and memorise individual fares.


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = forest().fit(X_train, y_train)
print(f"train accuracy   : {model.score(X_train, y_train):.4f}   <- always optimistic")
print(f"hold-out accuracy: {model.score(X_test, y_test):.4f}\n")

importance = sorted(zip(X.columns, model.feature_importances_), key=lambda t: -t[1])
for col, imp in importance:
    print(f"  {col:12s} {imp:.3f}  {'#' * int(imp * 100)}")

train accuracy   : 0.9017   <- always optimistic
hold-out accuracy: 0.8268

  Title        0.237  #######################
  Sex          0.195  ###################
  FarePerPax   0.146  ##############
  Fare         0.146  ##############
  Age          0.111  ###########
  Pclass       0.087  ########
  FamilySize   0.061  ######
  IsChild      0.011  #
  IsAlone      0.007  


## 5. The catch: good cross-validation, bad leaderboard

|                | cross-validation | Kaggle |
|---|---|---|
| sex alone      | 0.7867 | 0.7655 |
| random forest  | 0.8395 | **0.7608** |

Cross-validation rated the forest 5 points above the trivial baseline. On the
real test set it scored *below* it. Of the 73 rows where it overruled "women
live, men die", only 40% were right — coin flips.

So instead of trusting overall accuracy, we measured **each individual
override**: how often does this specific change beat the baseline? Three
corrections survived, and they use the family signal directly.

`family.py` finds relatives by `(Surname, Fare)` and by `Ticket`, then reports
whether a relative's outcome is **already known from `train.csv`** — true for
171 of the 418 test passengers. A row never sees its own label.

| correction | support in train |
|---|---|
| 3rd-class woman whose family is known to have died → dies | 87% of 55 rows |
| 3rd-class woman in a family of 5+ → dies | only 3 of 27 survived |
| `Master` (a boy) whose family lived → survives | 17 of 18 rows |

Restricting the first rule to **third class** is what took the score from
0.78468 to 0.80861: 1st- and 2nd-class women survived at 97% and 92% whatever
happened to their families.


In [ ]:
import numpy as np
from family import combine, family_survival
import rules

both, labels, n_train = combine(train_clean, test)
known = np.zeros(len(both), bool)
known[X_train.index] = True
fam = family_survival(both, labels, known)

rows = train_clean.loc[X_test.index]
sex_only  = (rows["Sex"] == "female").astype(int).values
rule_pred = rules.predict(rows, fam[:n_train][X_test.index])

print(f"on the same {len(y_test)} held-out rows:")
print(f"  sex baseline  : {accuracy_score(y_test, sex_only):.4f}")
print(f"  random forest : {accuracy_score(y_test, model.predict(X_test)):.4f}")
print(f"  rules         : {accuracy_score(y_test, rule_pred):.4f}   <- submitted")

on the same 179 held-out rows:
  sex baseline  : 0.7821


  random forest : 0.8268
  rules         : 0.8324   <- submitted


## 6. Prediction on the test set

Same rules, now with every one of the 891 training labels available.
Writes `submissions/submission.csv` — the file uploaded to Kaggle.


In [7]:
from predict import make_submission

submission = make_submission(train_clean, test)
print()
print(submission.head().to_string(index=False))
print(f"\npredicted survivors: {submission['Survived'].sum()} of {len(submission)}")

submission saved to /home/amine/Desktop/kaggle-titanic/submissions/submission.csv (418 rows, 171 with a known relative)

 PassengerId  Survived
         892         0
         893         1
         894         0
         895         0
         896         1

predicted survivors: 144 of 418


## What we learned

On 891 rows, cross-validation could not tell a real improvement from noise, and
it was optimistic by 8 points for the model that overfit hardest.

What actually tracked the leaderboard was the **number of bets placed against a
simple, strong baseline, and the measured hit rate of each one** — 73 bets at
40% lost points, 28 bets at 82% won them. Every improvement came from making
*fewer* but better-supported changes, never from a stronger model.
